# Bài 11: Máy nhìn
**Mở hộp đen AI · Tập 2 · Phòng thí nghiệm Colab**

Bốn chặng: **Chạy → Đọc → Sửa → Tự làm**. Phần lõi chỉ dùng NumPy và scikit-learn. Hai ô cuối là phần mở rộng, **kết quả tuỳ máy**.
Ebook Bên trong cỗ máy Trí tuệ nhân tạo - Tập 2. Phát hành tại https://montinhoc.vn

## Chặng 1: Chạy
### Ô 1. Gặp lại bảng số của một bức ảnh (Bài 2, Tập 1)

In [ ]:
import numpy as np
from sklearn.datasets import load_digits

so = load_digits()
anh = so.images[7]                      # ảnh thứ 8 trong bộ dữ liệu
print("Đây là chữ số", so.target[7])
print(anh.astype(int))

### Ô 2. Tích chập: trượt một bộ lọc 3 × 3 trên ảnh

In [ ]:
def tich_chap(anh, loc):
    cao, rong = anh.shape[0] - 2, anh.shape[1] - 2
    kq = np.zeros((cao, rong))
    for i in range(cao):
        for j in range(rong):
            o = anh[i:i + 3, j:j + 3]          # ô 3 x 3 đang xét
            kq[i, j] = np.sum(o * loc)         # nhân từng cặp rồi cộng
    return kq

LOC_DOC = np.array([[-1, 0, 1],
                    [-1, 0, 1],
                    [-1, 0, 1]])               # tìm cạnh dọc
print(tich_chap(anh, LOC_DOC).astype(int))

### Ô 3. Gộp cực đại: giữ lại tín hiệu mạnh nhất trong mỗi ô 2 × 2

In [ ]:
def relu(x):
    return np.maximum(x, 0)                    # số âm thành 0

def gop_cuc_dai(x):
    h, w = x.shape[0] // 2, x.shape[1] // 2
    return x[:2 * h, :2 * w].reshape(h, 2, w, 2).max(axis=(1, 3))

ban_do = gop_cuc_dai(relu(tich_chap(anh, LOC_DOC)))
print(ban_do.astype(int))
print("Kích thước: 8 x 8 -> 6 x 6 -> 3 x 3")

### Ô 4. Bốn bộ lọc làm 'đôi mắt' cho máy

In [ ]:
LOC = [LOC_DOC, LOC_DOC.T,                                     # dọc, ngang
       np.array([[0, 1, 1], [-1, 0, 1], [-1, -1, 0]]),            # chéo /
       np.array([[1, 1, 0], [1, 0, -1], [0, -1, -1]])]            # chéo \

def dac_trung_tich_chap(anh):
    ban_do = [gop_cuc_dai(relu(tich_chap(anh, k))).ravel() for k in LOC]
    return np.concatenate(ban_do)

X_tc = np.array([dac_trung_tich_chap(a) for a in so.images])
print("Mỗi ảnh: 64 điểm ảnh ->", X_tc.shape[1], "đặc trưng tích chập")

### Ô 5. So hai cách nhìn: điểm ảnh thô và đặc trưng tích chập

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

chi_so = np.arange(len(so.images))
i_hoc, i_thi = train_test_split(chi_so, test_size=0.3, random_state=0)
X_tho = so.data
y = so.target

mh_tho = LogisticRegression(max_iter=2000).fit(X_tho[i_hoc], y[i_hoc])
mh_tc = LogisticRegression(max_iter=2000).fit(X_tc[i_hoc], y[i_hoc])
print("Điểm ảnh thô     :", round(mh_tho.score(X_tho[i_thi], y[i_thi]), 3))
print("Đặc trưng tích chập:", round(mh_tc.score(X_tc[i_thi], y[i_thi]), 3))

## Chặng 2: Đọc
1. Ở ô 2, những ô nào trong kết quả có số dương lớn, số âm lớn? Chúng nằm ở đâu trên chữ số?
2. Ở ô 3, vì sao kết quả nhỏ hơn ảnh gốc? Mất thông tin gì, giữ được gì?
3. Ở ô 5, cách nào dùng nhiều đặc trưng hơn? Cách nào đúng hơn?

## Chặng 3: Sửa
### Ô 6. Dịch mọi ảnh thi sang phải 1 điểm ảnh

In [ ]:
def dich_phai(a, buoc=1):
    kq = np.zeros_like(a)
    kq[:, buoc:] = a[:, :-buoc]
    return kq

anh_dich = np.array([dich_phai(so.images[i]) for i in i_thi])
X_tho_dich = anh_dich.reshape(len(i_thi), -1)
X_tc_dich = np.array([dac_trung_tich_chap(a) for a in anh_dich])
print("Sau khi dịch 1 điểm ảnh:")
print("  Điểm ảnh thô     :", round(mh_tho.score(X_tho_dich, y[i_thi]), 3))
print("  Đặc trưng tích chập:", round(mh_tc.score(X_tc_dich, y[i_thi]), 3))

## Mở rộng (kết quả tuỳ máy)
### Ô 7. Để máy tự học bộ lọc: một mạng tích chập nhỏ bằng PyTorch
Colab có sẵn PyTorch. Kết quả mỗi lần chạy có thể hơi khác nhau.

In [ ]:
import torch, torch.nn as nn

Xh = torch.tensor(so.images[i_hoc], dtype=torch.float32).unsqueeze(1) / 16
Xt = torch.tensor(so.images[i_thi], dtype=torch.float32).unsqueeze(1) / 16
yh, yt = torch.tensor(y[i_hoc]), torch.tensor(y[i_thi])

mang = nn.Sequential(nn.Conv2d(1, 8, 3),            # 8 bộ lọc 3 x 3 tự học
                     nn.ReLU(), nn.MaxPool2d(2),
                     nn.Flatten(), nn.Linear(8 * 3 * 3, 10))
toi_uu = torch.optim.Adam(mang.parameters(), lr=0.01)
for vong in range(200):
    toi_uu.zero_grad()
    mat_mat = nn.functional.cross_entropy(mang(Xh), yh)
    mat_mat.backward()                     # lan truyền ngược (Bài 10)
    toi_uu.step()
print("Mạng tích chập tự học - đúng trên đề thi:",
      (mang(Xt).argmax(1) == yt).float().mean().item())

### Ô 8. Học chuyển giao với ảnh em tự chụp
Tải 20 ảnh của hai loại đồ vật (ví dụ bút và thước) vào hai thư mục `but/` và `thuoc/`, rồi dùng một mạng đã học sẵn trên ImageNet làm 'đôi mắt'.

In [ ]:
from torchvision import models, transforms
from PIL import Image
import glob

mat = models.resnet18(weights="DEFAULT"); mat.fc = nn.Identity(); mat.eval()
bien_doi = transforms.Compose([transforms.Resize((224, 224)),
                               transforms.ToTensor()])
X, nhan = [], []
for lop in ["but", "thuoc"]:
    for f in glob.glob(f"{lop}/*.jpg"):
        with torch.no_grad():
            anh = bien_doi(Image.open(f).convert("RGB")).unsqueeze(0)
            X.append(mat(anh)[0].numpy())
        nhan.append(lop)
Xh, Xt, yh, yt = train_test_split(np.array(X), nhan, test_size=0.3, random_state=0)
mh = LogisticRegression(max_iter=2000).fit(Xh, yh)
print("Học chuyển giao - đúng trên đề thi:", mh.score(Xt, yt))

## Chặng 4: Tự làm
1. Ở ô 6, dịch 2 điểm ảnh thay vì 1. Cách nào chịu được tốt hơn?
2. Tự thiết kế một bộ lọc 3 × 3 mới (ví dụ tìm góc), thêm vào danh sách `LOC` ở ô 4, chạy lại ô 5 và ô 6.
3. (Mở rộng) Chạy ô 7 trên Colab. Mạng tự học bộ lọc có tốt hơn bốn bộ lọc em đặt bằng tay không?

**Nhật ký dùng AI:** Em có nhờ AI ở bước nào không? Nó sai ở đâu? Em sửa thế nào?

*(ghi vào đây)*